In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)
library_path

In [ ]:
# load data
DATA_PATH = library_path.parent / "data"
PLOTS_PATH = library_path.parent / "plots"

df = pd.read_csv(f"{DATA_PATH}/all_data.csv", sep="\t")

In [ ]:
df.head()

In [ ]:
cols_to_use = ["sPCI", "pPCI"]
df = df[cols_to_use].copy()

In [ ]:
df.info()

In [ ]:

# Descriptive statistics
sPCI = df['sPCI'].to_numpy()
pPCI = df['pPCI'].to_numpy()

print(f"n = {len(sPCI)}")
print(f"sPCI:  mean = {sPCI.mean():.3f}, SD = {sPCI.std():.3f}, skew = {pd.Series(sPCI).skew():.3f}")
print(f"pPCI:  mean = {pPCI.mean():.3f}, SD = {pPCI.std():.3f}, skew = {pd.Series(pPCI).skew():.3f}")
print(f"mean difference (sPCI - pPCI) = {(sPCI - pPCI).mean():.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

axes[0].hist(df['sPCI'].dropna(), bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('sPCI')
axes[0].set_xlabel('sPCI')
axes[0].set_ylabel('Count')

axes[1].hist(df['pPCI'].dropna(), bins=20, color='salmon', edgecolor='white')
axes[1].set_title('pPCI')
axes[1].set_xlabel('pPCI')

plt.suptitle('Histograms of sPCI and pPCI')
plt.tight_layout()
plt.savefig(PLOTS_PATH / "histograms_sPCI_pPCI.png", dpi=600)
plt.show()


In [ ]:
from survival.agreement import paired_permutation_test, paired_permutation_test_median, paired_bootstrap_ci, sign_test, paired_bootstrap_ci_bca
from survival.agreement import bland_altman_analysis, intraclass_correlation, icc_bias_corrected, bland_altman_analysis_robust

In [ ]:
# Median permutation test
stat_med, p_med = paired_permutation_test_median(sPCI, pPCI, random_state=42)
print("Median test:", stat_med, "p_value:", p_med)

# Bootstrap CI for median
ci_med = paired_bootstrap_ci(sPCI, pPCI, statistic="median", random_state=42)
ci_med_bca = paired_bootstrap_ci_bca(sPCI, pPCI, statistic="median", random_state=42)

print("Median CI:", np.round(ci_med,3))
print("Median BCa CI:", np.round(ci_med_bca,3))

In [ ]:
stat, p_perm = paired_permutation_test(sPCI, pPCI, n_permutations=20000, random_state=42)
print("Permutation test:", stat, p_perm)

# Bootstrap CI for mean (optional)
ci_mean = paired_bootstrap_ci(sPCI, pPCI, statistic="mean", random_state=42)
ci_bca_mean = paired_bootstrap_ci_bca(sPCI, pPCI, statistic="mean", random_state=42)
print("Mean CI:", np.round(ci_mean,3))
print("Mean BCa CI:", np.round(ci_bca_mean,3))

In [ ]:
# Sign test
n_pos, n, p_sign = sign_test(sPCI, pPCI)
print("Sign test:", n_pos, "/", n, "p =", p_sign)

In [ ]:
# Inspect bootstrap distribution
boot_stats = []

rng = np.random.default_rng(42)
d = sPCI - pPCI
n = len(d)

for _ in range(20000):
    sample = d[rng.integers(0, n, n)]
    boot_stats.append(np.median(sample))

unique, counts = np.unique(boot_stats, return_counts=True)
print(dict(zip(unique, counts)))

In [ ]:
# Bland-Altman plot
result = bland_altman_analysis(sPCI, pPCI, plots_path=PLOTS_PATH)


In [ ]:
#icc
icc = intraclass_correlation(sPCI, pPCI)
print(icc)

In [ ]:
delta, icc_corr = icc_bias_corrected(sPCI, pPCI, method="median")

print("Bias removed:", delta)
print(icc_corr)

In [ ]:
robust_res = bland_altman_analysis_robust(sPCI, pPCI, plots_path=PLOTS_PATH)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

abs_diff = np.abs(pPCI - sPCI)

axes[0].scatter(sPCI, abs_diff, color="steelblue", edgecolors="white", s=70)
axes[0].set_xlabel("sPCI", fontsize=12)
axes[0].set_ylabel("|pPCI − sPCI|", fontsize=12)
axes[0].set_title("Absolute difference vs sPCI", fontsize=13)

axes[1].scatter(pPCI, abs_diff, color="salmon", edgecolors="white", s=70)
axes[1].set_xlabel("pPCI", fontsize=12)
axes[1].set_ylabel("|pPCI − sPCI|", fontsize=12)
axes[1].set_title("Absolute difference vs pPCI", fontsize=13)

plt.tight_layout()
plt.show()